# Create Pipeline Package for segmentation

In this notebook, the main goal is to create a Pipeline with all of the contents that are necessary for the execution of the model on AI Inference Server.

## Creating a PythonComponent wrapper

In this step, we create a `PythonComponent` that executes the Python script we explained in notebook [20-CreateInferenceWrapper](./20-CreateInferenceWrapper.ipynb).\
To do so, we need to 
- collect our file resources,
- define component inputs and outputs,
- define the required Python environment.

The resources contain our Ultralytics YOLO model, the entrypoint python script [segmentation.py](../src/segmentation.py), and an additional helper script [imageset.py](../src/imageset.py). In order to make the Ultralytics python package work, we prepared a [requirements.txt](../src/requirements.txt) with no transitive dependencies, as explained in the [previous notebook](./20-CreateInferenceWrapper.ipynb).

In [ ]:
from simaticai import deployment

model_name = "yolo11n-seg.pt"

component = deployment.PythonComponent("segmentation", python_version="3.12")

component.add_resources("../src/", "segmentation.py")
component.add_resources("../models/", model_name)
component.set_entrypoint("segmentation.py")
component.set_requirements("../src/requirements.txt", no_deps=True)

component.add_input("vision_payload", "ImageSet")

component.add_output("result_image_set", "ImageSet")
component.add_output("iuid", "String")
component.add_output("areas", "String")

The `imageset.py` file should have been created by notebook 20. In case it did not run, we create this file again here.

In [ ]:
from simaticai.common.resources import copy_resource_to
copy_resource_to('ImageSet', '../src')

Now we can add it as a resource to our component.

In [ ]:
component.add_resources("../src/", "imageset.py")
component

## Creating a Pipeline from the component

Now we can use the component to create a Pipeline configuration. The Pipeline requires a list of components - which is a single component in this example - and a name. Providing a description is optional.

In [ ]:
pipeline = deployment.Pipeline.from_components([component], name="Segmentation", version="1", desc="Segmentation tutorial with Ultralytics YOLO.")
pipeline

## Build the Pipeline Package

This step creates the proper content in the defined target folder `packages` and creates the edge Pipeline Package as a zip file. The export method first validates the Pipeline and raises an error if it finds any problems.

In [ ]:
edge_package_path = pipeline.export('../packages')
edge_package_path

## Test the Pipeline Package locally

We suggest to test the Pipeline Package on your computer before deploying it to the AI Inference Server. It is possible to do so using notebook [40-TestPipelineLocally](40-TestPipelineLocally.ipynb).